# Phase 7 (Prior Case Retrieval) — c-1 decoder-only + BitFit

The c-1 **decoder** (LM head skipped) is used as a bi-encoder: query and candidate cases are cut into token windows, run through the decoder (right-padded, causal), pooled to window vectors (mean / max / last token) and compared (cosine / dot / maxsim). Training is contrastive (InfoNCE or margin) with in-batch and/or hard negatives. Output: top 7 candidate ids per test query (`prediction.json`), scored by Micro-F1@7.

BitFit: only the chosen **bias** terms of the decoder train (by group and block).

**Phase 7 technique list:** full FT, partial/layer-wise FT, QLoRA, LORMA, prefix, prompt, IA3, KD. BitFit is **not** on it, so this run is a comparison point, not one of the 3 required techniques.

Run every cell top to bottom. Everything you choose is in section **[H]** of Cell E1 (all left `None`). GradCache (`CHUNK = 4`) and gradient checkpointing are on by default to avoid running out of GPU memory; they do not change the maths.

## Cell 0 — Phase 1 imports, tokenizer class, model classes, HF login

In [1]:
# --- Phase 1 c-1 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
# --- Phase 1 c-1 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


In [3]:
# --- Phase 1 c-1 notebook, cell 21 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


## Cell A — find and load the Phase 1 c-1 model and tokenizer (fix ROOT to what the find command shows)

In [4]:
!find /kaggle/input -path "*phase-1-c-1*" \( -name "*.pt" -o -name "*.pkl" \) -exec ls -la {} \;


-rw-rw-rw- 1 root root 429235 Nov  7 07:14 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/bpe_tokenizer.pkl
-rw-rw-rw- 1 root root 429235 Nov  7 07:14 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl
-rw-rw-rw- 1 root root 1331870311 Nov  7 07:14 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/ckpt.pt
-rw-rw-rw- 1 root root 443954142 Nov  7 07:14 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/model_final.pt


In [5]:
# =====================================================================================
# CELL A — load the Phase 1 c-1 decoder and its tokenizer
# Expect <All keys matched successfully>, 110.98M params, vocab 10000, d_model 1024, 8 layers, max_seq_len 2048
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

# [c-1] checkpoint folder is legal-gpt2-final (c-6 was model_encdec); class is DecoderLM + LMConfig
tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| d_ff", cfg.d_ff, "| max_seq_len", cfg.max_seq_len,
      "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)


<All keys matched successfully>
params 110.979072 M | vocab 10000 | d_model 1024 | layers 8 | d_ff 4096 | max_seq_len 2048 | norm_first False | dropout 0.1


## Cell B — decoder features sanity check

In [6]:
# =====================================================================================
# CELL B — per-token vectors from the c-1 decoder + sanity check
# [c-1] c-1 has no encoder: the features are the decoder's last hidden states (after ln_f),
#       with the LM head skipped. Each token only sees the tokens to its LEFT (causal).
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

# Sanity check: the LM head on these features must give the full model's logits
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)


max diff 0.0 (1, 64, 1024)


## Cell C — Phase 7 data: queries + candidate pools per split

In [7]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 7 (Prior Case Retrieval) data: queries + candidate pools per split
#   Train/train.json + train_candidates.json, Dev/dev.json + dev_candidates.json,
#   Test/test.json + test_candidates.json. Each split's queries are ranked ONLY against
#   that split's own candidate pool. "text" = the case split into sentences (joined here).
# The exact JSON layout is not in the spec, so the reader accepts the usual layouts and
# prints what it found. Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, random
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P7_DIR = "Phase_7_PCR"
print("Phase 7 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_7" in f])

def get7(name):
    """Reads Phase_7_PCR/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P7_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

ID_FIELDS   = ("id", "query_id", "candidate_id", "case_id", "qid", "cid", "doc_id")
TEXT_FIELDS = ("text", "document", "case_text", "content", "judgment", "sentences")
REL_FIELDS  = ("relevant_candidates", "relevant_cases", "relevant", "positives", "gold", "labels")
CAND_FIELDS = ("candidates", "candidate_ids", "candidate_pool")   # only if a query lists its own pool

def norm_id(x):
    """IDs as strings. Integer IDs are zero-padded to 10 digits like the spec's example output."""
    if isinstance(x, (int, np.integer)):
        return f"{int(x):010d}"
    return str(x).strip()

def join_text(t):
    if isinstance(t, dict):                              # e.g. {"sentences": [...]}
        t = next((t[f] for f in TEXT_FIELDS if f in t), list(t.values()))
    if isinstance(t, (list, tuple)):
        return " ".join(str(s).strip() for s in t if str(s).strip())
    return str(t)

def _unwrap(raw):
    # {"data": [...]} / {"queries": {...}} -> inner; never unwrap {"<case id>": {...}}
    while (isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict))
           and not norm_id(next(iter(raw))).isdigit()):
        raw = next(iter(raw.values()))
    idf = next((f for f in ID_FIELDS if isinstance(raw, dict) and isinstance(raw.get(f), list)), None)
    if idf:                                              # columns layout {"id": [...], "text": [...], ...}
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw[idf]))]
    return raw

def as_cases(raw, name, is_query):
    """-> {case_id: {"text": str, "rel": [ids] or None, "cands": [ids] or None}} in file order."""
    raw = _unwrap(raw)
    items = list(raw.items()) if isinstance(raw, dict) else [(None, d) for d in raw]
    out, n_dup = {}, 0
    for k, d in items:
        if not isinstance(d, dict):                      # {id: "text"} or {id: [sentences]}
            d = {"text": d}
        idf = next((f for f in ID_FIELDS if f in d), None)
        cid = norm_id(d[idf] if idf else k)
        assert cid not in (None, "None"), f"{name}: record without an id (keys {list(d)})"
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"{name} {cid}: no text field (keys {list(d)})"
        rf = next((f for f in REL_FIELDS if f in d), None) if is_query else None
        cf = next((f for f in CAND_FIELDS if f in d), None) if is_query else None
        rec = {"text": join_text(d[tf]),
               "rel": [norm_id(x) for x in d[rf]] if rf and d[rf] is not None else None,
               "cands": [norm_id(x) for x in d[cf]] if cf and d[cf] is not None else None}
        if cid in out:
            n_dup += 1
            if out[cid]["text"] != rec["text"]:
                print(f"WARNING {name}: id {cid} repeated with different text, keeping the first")
            continue
        out[cid] = rec
    print(f"{name}: {len(items)} records -> {len(out)} cases (repeated ids skipped {n_dup})")
    return out

def _peek(raw, name):
    r = _unwrap(raw)
    first = r[0] if isinstance(r, list) else next(iter(r.values()))
    print(f"{name} layout: {type(r).__name__} of {len(r)} | first record:",
          list(first) if isinstance(first, dict) else type(first).__name__)

queries, pools = {}, {}
for split, folder in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
    _q = get7(f"{folder}/{split}.json"); _peek(_q, f"{split}.json")
    _c = get7(f"{folder}/{split}_candidates.json"); _peek(_c, f"{split}_candidates.json")
    queries[split] = as_cases(_q, f"{split} queries", True)
    pools[split] = as_cases(_c, f"{split} candidates", False)
del _q, _c

# test labels are private: ignore anything that looks like them
for d in queries["test"].values():
    d["rel"] = None

# Gold sets (train/dev): drop relevant ids that are not in the split's pool (cannot be retrieved)
GOLD = {}
for split in ("train", "dev"):
    pool, g, n_missing, n_rel = pools[split], {}, 0, 0
    for q, d in queries[split].items():
        rel = list(dict.fromkeys(d["rel"] or []))
        n_rel += len(rel)
        keep = [c for c in rel if c in pool and c != q]
        n_missing += len(rel) - len(keep)
        g[q] = set(keep)
    GOLD[split] = g
    n_empty = sum(not s for s in g.values())
    print(f"{split}: {n_rel} relevant links, {n_missing} not in the {split} pool (dropped) | "
          f"queries with no retrievable relevant case: {n_empty}")

# Queries with no gold cannot teach anything; they still count in the dev score (as the evaluator would)
TRAIN_QIDS = [q for q in queries["train"] if GOLD["train"][q]]

for split in ("train", "dev", "test"):
    Q, P = queries[split], pools[split]
    qw = np.array([len(d["text"].split()) for d in Q.values()])
    cw = np.array([len(d["text"].split()) for d in P.values()])
    line = (f"{split:5s} queries {len(Q):5d} | pool {len(P):6d} | query words median {np.median(qw):.0f} "
            f"p90 {np.percentile(qw, 90):.0f} | candidate words median {np.median(cw):.0f} "
            f"p90 {np.percentile(cw, 90):.0f}")
    if split in GOLD:
        gs = np.array([len(s) for s in GOLD[split].values()])
        line += f" | gold per query mean {gs.mean():.2f} median {np.median(gs):.0f} max {gs.max()} | >7: {(gs > 7).mean():.1%}"
    n_self = sum(q in P for q in Q)
    if n_self:
        line += f" | {n_self} queries are also in their own pool (excluded when ranking)"
    if any(d["cands"] for d in Q.values()):
        line += " | queries carry their own candidate lists (ranking restricted to them)"
    print(line)
print("pool overlap train∩dev", len(set(pools["train"]) & set(pools["dev"])),
      "| train∩test", len(set(pools["train"]) & set(pools["test"])),
      "| dev∩test", len(set(pools["dev"]) & set(pools["test"])))
print("first test query ids:", list(queries["test"])[:3])


Phase 7 files on HF: ['Phase_7_PCR/Dev/dev.json', 'Phase_7_PCR/Dev/dev_candidates.json', 'Phase_7_PCR/Test/test.json', 'Phase_7_PCR/Test/test_candidates.json', 'Phase_7_PCR/Train/train.json', 'Phase_7_PCR/Train/train_candidates.json']


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train.json layout: dict of 827 | first record: ['text', 'relevant_candidates']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

... (7 outputs left out)

test.json layout: dict of 237 | first record: ['text']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test_candidates.json layout: dict of 1727 | first record: ['text']
test queries: 237 records -> 237 cases (repeated ids skipped 0)
test candidates: 1727 records -> 1727 cases (repeated ids skipped 0)
train: 5308 relevant links, 0 not in the train pool (dropped) | queries with no retrievable relevant case: 10
dev: 973 relevant links, 0 not in the dev pool (dropped) | queries with no retrievable relevant case: 0
train queries   827 | pool   4320 | query words median 4508 p90 12602 | candidate words median 3632 p90 12422 | gold per query mean 6.42 median 4 max 52 | >7: 30.0% | 827 queries are also in their own pool (excluded when ranking)
dev   queries   118 | pool   1023 | query words median 4620 p90 15333 | candidate words median 4036 p90 16599 | gold per query mean 8.25 median 5 max 92 | >7: 33.1% | 118 queries are also in their own pool (excluded when ranking)
test  queries   237 | pool   1727 | query words median 4653 p90 14065 | candidate words median 3947 p90 13491 | 237 queries ar

## Cell D — tokenisation (cached), Micro-F1@7 scorer, prediction.json writer, BM25 baseline

In [8]:
# =====================================================================================
# CELL D — [ANCILLARY] tokenisation (cached), Micro-F1@7 scorer, prediction.json writer,
#          BM25 baseline (also used as a hard-negative source in Cell E2 if you choose it)
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import os, re, json, glob, time, zipfile, pickle, hashlib
import numpy as np
import scipy.sparse as sp
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
TOPK = 7                                 # the evaluator scores exactly the first 7 ids per query

# -------------------------------------------------------------------------------------
# Tokenise every query and candidate once with the c-1 BPE tokenizer -> d["ids"].
# The Python BPE is slow on long judgments, so: identical texts are encoded once, all CPU cores
# are used (fork), and the result is saved to /kaggle/working/p7_c1_tok.pkl. Attach a finished
# run's output as input next time and the cache is found and reused automatically.
# -------------------------------------------------------------------------------------
TOK_CACHE_OUT = "/kaggle/working/p7_c1_tok.pkl"

def _h(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def _enc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

_all_docs = [d for s in ("train", "dev", "test") for part in (queries[s], pools[s]) for d in part.values()]
_cache = {}
for _p in sorted(glob.glob("/kaggle/input/**/p7_c1_tok.pkl", recursive=True)) + [TOK_CACHE_OUT]:
    if os.path.exists(_p):
        _cache.update(pickle.load(open(_p, "rb")))
        print(f"token cache loaded from {_p}: {len(_cache)} texts")
_todo = list({_h(d["text"]): d["text"] for d in _all_docs if _h(d["text"]) not in _cache}.items())
print(f"{len(_all_docs)} documents, {len(_todo)} distinct texts still to tokenise")
if _todo:
    t0 = time.time()
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(os.cpu_count()) as _pool:
            _ids = list(tqdm(_pool.imap(_enc, [t for _, t in _todo], chunksize=4), total=len(_todo), desc="tokenising"))
    except Exception as e:                                       # no fork available: one process
        print("parallel tokenising failed, falling back to one process:", repr(e))
        _ids = [_enc(t) for _, t in tqdm(_todo, desc="tokenising")]
    _cache.update({h: ids for (h, _), ids in zip(_todo, _ids)})
    pickle.dump(_cache, open(TOK_CACHE_OUT, "wb"))
    print(f"tokenised in {(time.time() - t0) / 60:.1f} min, cache saved to {TOK_CACHE_OUT}")
for d in _all_docs:
    d["ids"] = _cache[_h(d["text"])]
del _cache, _todo

for s in ("train", "dev", "test"):
    for kind, part in (("queries", queries[s]), ("pool", pools[s])):
        L = np.array([len(d["ids"]) for d in part.values()])
        print(f"{s:5s} {kind:7s} c-1 tokens: median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
              f"max {L.max()} | windows of 512 needed (median) {np.median(np.ceil(L / 512)):.0f}")


# -------------------------------------------------------------------------------------
# Scorer: Micro-F1@7 (primary), Micro-P@7, Micro-R@7 as in the spec (gold not truncated or padded),
# plus ranking diagnostics that look past rank 7: Recall@100 and MRR.
#   pred: {qid: ranked list of candidate ids}, gold: {qid: set of ids}
# -------------------------------------------------------------------------------------
def evaluate_ranking(pred, gold, k=TOPK, show=True, name="dev"):
    tp = n_pred = n_gold = 0
    per_q, r100, rr = {}, [], []
    for q, g in gold.items():
        top = list(pred.get(q, []))[:k]
        hit = len(set(top) & g)
        tp += hit; n_pred += len(top); n_gold += len(g)
        per_q[q] = {"hits": hit, "gold": len(g), "f1": 2 * hit / (len(top) + len(g)) if (top or g) else 0.0}
        if g:
            full = list(pred.get(q, []))
            r100.append(len(set(full[:100]) & g) / len(g))
            first = next((i for i, c in enumerate(full) if c in g), None)
            rr.append(0.0 if first is None else 1.0 / (first + 1))
    P = tp / max(1, n_pred); R = tp / max(1, n_gold)
    F = 2 * P * R / (P + R) if P + R else 0.0
    res = {"micro_f1": F, "micro_p": P, "micro_r": R, "recall@100": float(np.mean(r100)) if r100 else 0.0,
           "mrr": float(np.mean(rr)) if rr else 0.0, "tp": tp, "per_query": per_q}
    if show:
        print(f"{name}: Micro-F1@{k} {F:.4f} | P@{k} {P:.4f} | R@{k} {R:.4f} | Recall@100 {res['recall@100']:.4f} "
              f"| MRR {res['mrr']:.4f} | queries {len(gold)}")
    return res


# -------------------------------------------------------------------------------------
# prediction.json writer: {query_id: [7 unique candidate ids from that query's test pool, best first]}
# for every test query, then submission.zip with prediction.json at the root.
# -------------------------------------------------------------------------------------
def allowed_pool(q, split):
    d = queries[split][q]
    ids = d["cands"] if d.get("cands") else pools[split].keys()
    return [c for c in ids if c != q and c in pools[split]]

def write_prediction(pred, out_dir="/kaggle/working", split="test"):
    os.makedirs(out_dir, exist_ok=True)
    out = {}
    for q in queries[split]:
        allowed = set(allowed_pool(q, split))
        top = [c for c in dict.fromkeys(pred.get(q, [])) if c in allowed][:TOPK]
        assert len(top) == TOPK, f"query {q}: only {len(top)} valid candidates"
        out[q] = top
    assert len(out) == len(queries[split])
    path = os.path.join(out_dir, "prediction.json")
    json.dump(out, open(path, "w"), indent=1)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {path} ({len(out)} queries x {TOPK}) and {zpath}")
    return out


# -------------------------------------------------------------------------------------
# BM25 over lower-cased words (k1 = 1.2, b = 0.75, the textbook defaults; not tuned).
# A lexical reference point for the report, and a source of hard negatives for training.
# -------------------------------------------------------------------------------------
_TOKRE = re.compile(r"[a-z0-9]+")

class BM25:
    def __init__(self, pool_docs, k1=1.2, b=0.75):
        self.ids = list(pool_docs)
        self.vocab = {}
        rows, cols, vals, dl = [], [], [], []
        for i, c in enumerate(tqdm(self.ids, desc="BM25 index", leave=False)):
            tf = Counter(_TOKRE.findall(pool_docs[c]["text"].lower()))
            dl.append(sum(tf.values()))
            for w, n in tf.items():
                rows.append(i); cols.append(self.vocab.setdefault(w, len(self.vocab))); vals.append(n)
        tf = sp.csr_matrix((np.array(vals, np.float32), (rows, cols)), shape=(len(self.ids), len(self.vocab)))
        dl = np.array(dl, np.float32)
        df = np.bincount(tf.indices, minlength=len(self.vocab))
        idf = np.log(1 + (len(self.ids) - df + 0.5) / (df + 0.5)).astype(np.float32)
        norm = k1 * (1 - b + b * dl / dl.mean())
        tf = tf.tocoo()
        w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col]
        self.W = sp.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape).T.tocsr()   # (V, Ndocs)

    def scores(self, text):
        q = Counter(w for w in _TOKRE.findall(text.lower()) if w in self.vocab)
        if not q:
            return np.zeros(len(self.ids), np.float32)
        qv = sp.csr_matrix((np.ones(len(q), np.float32), ([0] * len(q), [self.vocab[w] for w in q])),
                           shape=(1, len(self.vocab)))           # unique query terms (queries are whole judgments)
        return np.asarray((qv @ self.W).todense()).ravel()

def bm25_rank(split, top=200):
    """-> {qid: top `top` candidate ids of the split's pool by BM25}, self / non-allowed excluded."""
    bm = BM25(pools[split])
    pos = {c: i for i, c in enumerate(bm.ids)}
    out = {}
    for q, d in tqdm(queries[split].items(), desc=f"BM25 {split}"):
        s = bm.scores(d["text"])
        allowed = allowed_pool(q, split)
        idx = np.array([pos[c] for c in allowed])
        order = idx[np.argsort(-s[idx], kind="stable")[:top]]
        out[q] = [bm.ids[i] for i in order]
    return out

BM25_RANK = {"dev": bm25_rank("dev")}
print("\n-- BM25 baseline")
bm25_dev = evaluate_ranking(BM25_RANK["dev"], GOLD["dev"], name="BM25 dev")


8252 documents, 5819 distinct texts still to tokenise


tokenising:   0%|          | 0/5819 [00:00<?, ?it/s]

tokenised in 0.4 min, cache saved to /kaggle/working/p7_c1_tok.pkl
train queries c-1 tokens: median 6099 p90 16783 max 164014 | windows of 512 needed (median) 12
train pool    c-1 tokens: median 4851 p90 16601 max 551745 | windows of 512 needed (median) 10
dev   queries c-1 tokens: median 6204 p90 20392 max 143282 | windows of 512 needed (median) 12
dev   pool    c-1 tokens: median 5382 p90 22459 max 551745 | windows of 512 needed (median) 11
test  queries c-1 tokens: median 6263 p90 18436 max 118277 | windows of 512 needed (median) 13
test  pool    c-1 tokens: median 5260 p90 18505 max 551745 | windows of 512 needed (median) 11


BM25 index:   0%|          | 0/1023 [00:00<?, ?it/s]

BM25 dev:   0%|          | 0/118 [00:00<?, ?it/s]


-- BM25 baseline
BM25 dev: Micro-F1@7 0.1012 | P@7 0.1102 | R@7 0.0935 | Recall@100 0.4073 | MRR 0.3248 | queries 118


## Cell E0 — retrieval plumbing (c-1 bi-encoder, windows, similarity, losses, negatives, GradCache, training loop)

In [9]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared retrieval plumbing for the c-1 decoder-only Phase 7 (PCR) runs
# Run once after Cells 0, A, B, C, D. Then run Cell E1 (the method cell).
# Needs from earlier cells: lm, cfg, device, features, queries, pools, GOLD, TRAIN_QIDS, PAD_ID,
#   evaluate_ranking, allowed_pool, write_prediction, bm25_rank, BM25_RANK
#
# Same retrieval design as the c-6 Phase 7 notebook (only the backbone differs):
# the c-1 DECODER (LM head skipped) is used as a bi-encoder (shared weights for query and candidate):
#   case -> token windows -> DECODER last hidden states (after ln_f) -> one vector per window
#        (POOL) -> document = its window vectors; similarity(query, candidate) (SIMILARITY)
#   Candidates of a pool are encoded once, every query is scored against all of them.
#
# [c-1] Causal attention: token t only sees tokens <= t. So
#   - windows are padded on the RIGHT: real tokens never see padding, no mask is needed, and a
#     padded row gives exactly the same vectors as the window run alone (self-check below);
#   - POOL "last" (the last real token, the only one that has seen the whole window) is the
#     decoder analogue of a [CLS] vector; "mean" / "max" pool over the real tokens only.
#     ("first" is not offered: the first token of a causal model sees only itself.)
#   The decoder blocks are the SAME modules with the SAME weights (AdaLoRA wrappers included).
#
# Long documents (position table has 2048 rows, judgments are far longer): windows of WINDOW
#   tokens every STRIDE tokens (Phase 2-6 make_windows), DOC_REPR picks which ones stand for
#   the case ("all", "first", "last", "first_last", "spread"), up to MAX_WINDOWS.
#
# Similarity (choice in E1):
#   "cosine" / "dot" : window vectors -> one case vector (WIN_AGG "mean" or "max"), then cosine / dot
#   "maxsim"         : late interaction on L2-normalised window vectors: for every query window
#                      take its best-matching candidate window, average over query windows
#
# Training objective (choice in E1), one row per query with one sampled relevant candidate:
#   "infonce" : softmax cross-entropy over [its positive, its hard negatives, and (IN_BATCH_NEG)
#               every other candidate in the batch], logits = similarity / TAU
#   "margin"  : pairwise hinge  mean(max(0, MARGIN - s(q, pos) + s(q, neg))) over the same negatives
#   Candidates relevant to the query (other than the sampled positive) are never used as its
#   negatives (multi-relevant queries).
# Hard negatives (choice in E1): "none", "random" (from the pool), "bm25" (BM25 top HARD_TOP,
#   non-relevant), "model" (current model's top HARD_TOP, re-mined at the start of every epoch).
#
# Memory: CHUNK > 0 turns on gradient caching (GradCache): same loss and gradients as one big
#   batch, encoded a chunk at a time. Training hooks (the AdaLoRA allocator) are called with
#   unscaled gradients after every backward and after every optimizer step.
#
# Also here: the AdaLoRA rank allocator (your Phase 3 c-5 / Phase 6 c-1 AdaLoRAAllocator, PEFT
# RankAllocator logic), as a training hook.
# Nothing in this cell is a model design or a hyperparameter choice: every choice is an argument
# set in Cell E1.
# =====================================================================================
import math, time, random, os, json
import numpy as np
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model          # 1024 for c-1
MAX_POS = cfg.max_seq_len       # 2048: longest window
GRAD_CHECKPOINT = False         # set in Cell E1
_AUTOCAST = torch.cuda.is_available()


# -------------------------------------------------------------------------------------
# Decoder path: CausalSelfAttention / Block / DecoderLM.forward, same maths, attention in
# PyTorch's fused scaled_dot_product_attention with the Phase 1 scale (your Phase 6 c-1 E0)
# -------------------------------------------------------------------------------------
def _attn(attn, x):
    B, T, C = x.shape
    H, D = attn.n_heads, attn.d_head
    q, k, v = [p(x).view(B, T, H, D).transpose(1, 2) for p in (attn.q_proj, attn.k_proj, attn.v_proj)]
    dp = attn.attn_dropout.p if attn.training else 0.0
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp, is_causal=True, scale=attn.attention_scale)
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

def _block(blk, x):
    if blk.norm_first:                        # Block.forward, pre-norm
        x = x + _attn(blk.attn, blk.ln1(x))
        return x + blk.ffn(blk.ln2(x))
    x = blk.ln1(x + _attn(blk.attn, x))       # Block.forward, post-norm
    return blk.ln2(x + blk.ffn(x))

def hidden(backbone, idx):
    """idx (B, T), right-padded -> last hidden states (B, T, 1024) after ln_f (LM head not applied)."""
    T = idx.size(1)
    assert T <= MAX_POS, f"input of {T} tokens > {MAX_POS}"
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        x = x + backbone.pos_emb[:T, :]
    ck = GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training
    for blk in backbone.blocks:
        x = checkpoint(_block, blk, x, use_reentrant=False) if ck else _block(blk, x)
    return backbone.ln_f(x)


# -------------------------------------------------------------------------------------
# Which linears can be wrapped, by short name (same names in all 8 blocks)
# -------------------------------------------------------------------------------------
_WHERE = {"q_proj": ("attn", "q_proj"), "k_proj": ("attn", "k_proj"), "v_proj": ("attn", "v_proj"),
          "out_proj": ("attn", "out_proj"), "ffn_in": ("ffn", "net", "0"), "ffn_out": ("ffn", "net", "2")}
ALL_TARGETS = tuple(_WHERE)

def all_target_names(backbone=None):
    return ALL_TARGETS

def model_linears(backbone, which):
    """Yield (parent, attr, module) for each chosen linear, block by block, in `which` order."""
    for w in which:
        assert w in _WHERE, f"unknown target {w}; choose from {ALL_TARGETS}"
    for blk in backbone.blocks:
        for w in which:
            parent = blk
            for p in _WHERE[w][:-1]:
                parent = getattr(parent, p)
            yield parent, _WHERE[w][-1], getattr(parent, _WHERE[w][-1])

def wrap_linears(backbone, which, make_wrapper):
    todo = list(model_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# AdaLoRA rank allocator (your Phase 3 c-5 / Phase 6 c-1 AdaLoRAAllocator, unchanged), as a
# training hook. Works with any layer that has ada_P (out, r), ada_E (r,), ada_Q (r, in),
# an ada_mask buffer (r,) and orth_penalty().
#   - importance of every P / E / Q entry: EMA of |theta * grad| (beta1) times the EMA of its
#     deviation (uncertainty, beta2); a triplet's score = E score + mean P column + mean Q row
#   - budget (total kept triplets over ALL layers): init_rank x layers for the first tinit
#     steps, cubic decrease to target_rank x layers, then fixed for the last tfinal steps
#   - every delta_t steps (and every step in the final phase): keep the top-`budget` triplets
#     over all layers, E = 0 for the rest; in the final phase the pattern is frozen
#   - extra_loss() = orth_reg x mean orthogonality penalty, added to the training loss
# -------------------------------------------------------------------------------------
class AdaLoRAAllocator:
    def __init__(self, layers, init_rank, target_rank, total_steps, tinit, tfinal, delta_t, beta1, beta2, orth_reg):
        assert tinit < total_steps - tfinal, "warm-up and final phase leave no steps for pruning"
        self.layers = layers
        self.init_budget, self.target_budget = init_rank * len(layers), target_rank * len(layers)
        self.T, self.tinit, self.tfinal, self.dT = total_steps, tinit, tfinal, delta_t
        self.b1, self.b2, self.orth_reg = beta1, beta2, orth_reg
        self.ipt, self.unc = {}, {}
        self.step, self.budget, self.frozen = 0, self.init_budget, False

    def _params(self, m):
        return {"P": m.ada_P, "E": m.ada_E, "Q": m.ada_Q}

    @torch.no_grad()
    def after_backward(self):
        if self.step >= self.T - self.tfinal:
            return
        for m in self.layers:
            for p in self._params(m).values():
                if p.grad is None or not torch.isfinite(p.grad).all():
                    return                       # fp16 overflow step: GradScaler skips it, so do we
        for li, m in enumerate(self.layers):
            for k, p in self._params(m).items():
                s = (p * p.grad).abs().float()
                key = (li, k)
                if key not in self.ipt:
                    self.ipt[key], self.unc[key] = torch.zeros_like(s), torch.zeros_like(s)
                self.ipt[key].mul_(self.b1).add_((1 - self.b1) * s)
                self.unc[key].mul_(self.b2).add_((1 - self.b2) * (s - self.ipt[key]).abs())

    def _triplet_scores(self, li):
        sc = {k: self.ipt[(li, k)] * self.unc[(li, k)] for k in ("P", "E", "Q")}
        return sc["E"] + sc["P"].mean(0) + sc["Q"].mean(1)   # (rank,)

    def _schedule(self):
        t = self.step
        if t <= self.tinit:
            return self.init_budget, False
        if t > self.T - self.tfinal:
            return self.target_budget, True
        mul = 1 - (t - self.tinit) / (self.T - self.tfinal - self.tinit)
        b = int((self.init_budget - self.target_budget) * mul ** 3 + self.target_budget)
        return b, t % self.dT == 0

    @torch.no_grad()
    def after_step(self):
        self.step += 1
        self.budget, do_mask = self._schedule()
        if not do_mask or not self.ipt:
            return
        if not self.frozen:
            scores = [self._triplet_scores(li) for li in range(len(self.layers))]
            flat = torch.cat(scores)
            keep = torch.zeros_like(flat, dtype=torch.bool)
            keep[flat.topk(min(self.budget, flat.numel())).indices] = True
            for m, kp in zip(self.layers, keep.split([s.numel() for s in scores])):
                m.ada_mask.copy_(kp.float())
            if self.step > self.T - self.tfinal:
                self.frozen = True
        for m in self.layers:
            m.ada_E.mul_(m.ada_mask)

    def extra_loss(self):
        return self.orth_reg * sum(m.orth_penalty() for m in self.layers) / len(self.layers)

    def postfix(self):
        return {"budget": self.budget}

    def rank_pattern(self):
        """Ranks actually used per layer = non-zero E entries (works after loading saved weights).
        Between re-allocation steps a pruned triplet can get a small non-zero E again from the
        optimizer (same as PEFT); in the final phase E is re-masked every step."""
        return [int((m.ada_E != 0).sum()) for m in self.layers]

    def mask_pattern(self):
        """Ranks per layer the allocator last decided (ada_mask), i.e. the allocation itself."""
        return [int(m.ada_mask.sum()) for m in self.layers]

    def state_dict(self):
        return {"ipt": {k: v.cpu() for k, v in self.ipt.items()}, "unc": {k: v.cpu() for k, v in self.unc.items()},
                "step": self.step, "budget": self.budget, "frozen": self.frozen,
                "masks": [m.ada_mask.cpu() for m in self.layers]}

    def load_state_dict(self, s):
        dev = self.layers[0].ada_E.device
        self.ipt = {k: v.to(dev) for k, v in s["ipt"].items()}
        self.unc = {k: v.to(dev) for k, v in s["unc"].items()}
        self.step, self.budget, self.frozen = s["step"], s["budget"], s["frozen"]
        for m, mk in zip(self.layers, s["masks"]):
            m.ada_mask.copy_(mk.to(dev))


# -------------------------------------------------------------------------------------
# Windows (Phase 2-6 make_windows + select_windows, unchanged)
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")
POOLS = ("mean", "max", "last")
WIN_AGGS = ("mean", "max")
SIMILARITIES = ("cosine", "dot", "maxsim")
LOSSES = ("infonce", "margin")
HARD_SOURCES = ("none", "random", "bm25", "model")
POS_SAMPLINGS = ("one_per_query", "all_pairs")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]

def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)

def build_pieces(docs, window, stride, doc_repr, max_windows, desc="windows"):
    """{case_id: d} -> {case_id: [token lists, one per selected window]}."""
    assert window <= MAX_POS and stride <= window and doc_repr in DOC_REPRS
    assert doc_repr == "all" or max_windows, "set MAX_WINDOWS for this DOC_REPR"
    out = {k: [d["ids"][s:e] for s, e in select_windows(make_windows(len(d["ids"]), window, stride),
                                                           doc_repr, max_windows)]
           for k, d in tqdm(docs.items(), desc=desc, leave=False)}
    nw = np.array([len(v) for v in out.values()])
    seen = sum(sum(map(len, v)) for v in out.values()) / max(1, sum(len(d["ids"]) for d in docs.values()))
    print(f"{desc}: {len(out)} cases | windows per case mean {nw.mean():.2f} max {nw.max()} | tokens seen {seen:.1%}")
    return out


# -------------------------------------------------------------------------------------
# Bi-encoder: list of cases (each a list of windows) -> window vectors E (N, W, D) + mask M (N, W)
# -------------------------------------------------------------------------------------
def _pad(rows):
    """[c-1] rows padded on the RIGHT (causal: real tokens never see the padding, no mask needed)."""
    T = max(len(r) for r in rows)
    idx = torch.tensor([r + [PAD_ID] * (T - len(r)) for r in rows], dtype=torch.long, device=device)
    L = torch.tensor([len(r) for r in rows], device=device)
    mask = torch.arange(T, device=device)[None, :] < L[:, None]
    return idx, None, mask

class PCREncoder(nn.Module):
    def __init__(self, backbone, pool):
        super().__init__()
        assert pool in POOLS
        self.backbone, self.pool = backbone, pool

    def forward(self, cases):
        flat = [w for c in cases for w in c]
        idx, key_mask, tmask = _pad(flat)
        h = hidden(self.backbone, idx)                               # (Nw, T, D), causal, right-padded
        m = tmask.unsqueeze(-1).to(h.dtype)
        if self.pool == "mean":
            v = (h * m).sum(1) / m.sum(1).clamp_min(1)
        elif self.pool == "max":
            v = h.masked_fill(~tmask.unsqueeze(-1), torch.finfo(h.dtype).min).max(1).values
        else:                                                        # "last": last real token of each window
            v = h[torch.arange(h.size(0), device=h.device), tmask.sum(1) - 1]
        W = max(len(c) for c in cases)
        E = v.new_zeros(len(cases), W, v.size(-1))
        M = torch.zeros(len(cases), W, dtype=torch.bool, device=v.device)
        i = 0
        for r, c in enumerate(cases):
            E[r, :len(c)] = v[i:i + len(c)]; M[r, :len(c)] = True
            i += len(c)
        return E, M


def similarity(qE, qM, cE, cM, sim, win_agg):
    """(Nq, Wq, D), (Nq, Wq), (Nc, Wc, D), (Nc, Wc) -> scores (Nq, Nc), fp32."""
    qE, cE = qE.float(), cE.float()
    if sim == "maxsim":
        qn, cn = F.normalize(qE, dim=-1), F.normalize(cE, dim=-1)
        s = torch.einsum("qid,cjd->qcij", qn, cn)                    # window-to-window cosine
        s = s.masked_fill(~cM[None, :, None, :], -1e4).max(-1).values  # best candidate window
        qm = qM[:, None, :].float()
        return (s * qm).sum(-1) / qm.sum(-1).clamp_min(1)            # mean over query windows
    def agg(E, M):
        if win_agg == "mean":
            m = M.unsqueeze(-1).float()
            return (E * m).sum(1) / m.sum(1).clamp_min(1)
        return E.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
    qv, cv = agg(qE, qM), agg(cE, cM)
    if sim == "cosine":
        qv, cv = F.normalize(qv, dim=-1), F.normalize(cv, dim=-1)
    return qv @ cv.T


# -------------------------------------------------------------------------------------
# Encoding a whole pool, ranking, prediction
# -------------------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(enc, pieces, ids, batch_windows, desc="encode"):
    """-> E (N, Wmax, D) fp16, M (N, Wmax) for `ids` in order. Batches by window count (less padding)."""
    enc.eval()
    order = sorted(range(len(ids)), key=lambda i: sum(map(len, pieces[ids[i]])))
    W = max(len(pieces[k]) for k in ids)
    E = torch.zeros(len(ids), W, FEAT_DIM, dtype=torch.float16, device=device)
    M = torch.zeros(len(ids), W, dtype=torch.bool, device=device)
    i, pbar = 0, tqdm(total=len(ids), desc=desc, leave=False)
    while i < len(order):
        j, nw = i, 0
        while j < len(order) and (j == i or nw + len(pieces[ids[order[j]]]) <= batch_windows):
            nw += len(pieces[ids[order[j]]]); j += 1
        sel = order[i:j]
        with torch.autocast("cuda", dtype=torch.float16, enabled=_AUTOCAST):
            e, m = enc([pieces[ids[t]] for t in sel])
        E[sel, :e.size(1)] = e.half(); M[sel, :m.size(1)] = m
        pbar.update(j - i); i = j
    pbar.close()
    return E, M

@torch.no_grad()
def rank_split(enc, split, q_pieces, c_pieces, sim, win_agg, batch_windows, top=200, q_ids=None, return_scores=False):
    """-> {qid: top `top` allowed candidate ids, best first} (and their scores if return_scores)."""
    cids = list(pools[split])
    pos = {c: i for i, c in enumerate(cids)}
    q_ids = list(queries[split]) if q_ids is None else q_ids
    cE, cM = encode_cases(enc, c_pieces, cids, batch_windows, desc=f"encode {split} pool")
    qE, qM = encode_cases(enc, q_pieces, q_ids, batch_windows, desc=f"encode {split} queries")
    out, sc = {}, {}
    qb = 8 if sim == "maxsim" else 256
    cb = 2048 if sim == "maxsim" else len(cids)
    for s in tqdm(range(0, len(q_ids), qb), desc=f"rank {split}", leave=False):
        S = torch.cat([similarity(qE[s:s + qb], qM[s:s + qb], cE[t:t + cb], cM[t:t + cb], sim, win_agg)
                       for t in range(0, len(cids), cb)], dim=1)
        for r, q in enumerate(q_ids[s:s + qb]):
            row = S[r]
            d = queries[split][q]
            if d.get("cands") or q in pos:                           # restrict to allowed candidates
                keep = torch.zeros_like(row, dtype=torch.bool)
                keep[[pos[c] for c in allowed_pool(q, split)]] = True
                row = row.masked_fill(~keep, -float("inf"))
            v, i = torch.topk(row, min(top, row.numel()))
            ok = torch.isfinite(v)
            out[q] = [cids[j] for j in i[ok].tolist()]
            sc[q] = v[ok].tolist()
    del cE, cM, qE, qM
    return (out, sc) if return_scores else out


# -------------------------------------------------------------------------------------
# Training batches
# -------------------------------------------------------------------------------------
def make_epoch_rows(pos_sampling, gold):
    """-> list of (qid, positive cid) for one epoch, shuffled."""
    rows = []
    for q in TRAIN_QIDS:
        g = sorted(gold[q])
        rows += [(q, random.choice(g))] if pos_sampling == "one_per_query" else [(q, c) for c in g]
    random.shuffle(rows)
    return rows

def pick_negatives(q, n_hard, source, hard_rank, hard_top, gold):
    if not n_hard or source == "none":
        return []
    if source == "random":
        pool_ids = TRAIN_POOL_IDS
        out = set()
        while len(out) < n_hard:
            c = random.choice(pool_ids)
            if c != q and c not in gold[q]:
                out.add(c)
        return list(out)
    cands = [c for c in hard_rank[q][:hard_top] if c not in gold[q] and c != q]
    return random.sample(cands, min(n_hard, len(cands)))

def batch_mask(rows, cand_ids, own_negs, gold, in_batch, dev):
    """rows[i] = (qid, pos); cand_ids = unique candidate ids in the batch (columns);
    own_negs[i] = set of column indices that are query i's own hard negatives.
    -> allowed (B, C) bool (positive + usable negatives), target (B,) column of each positive."""
    col = {c: j for j, c in enumerate(cand_ids)}
    B, C = len(rows), len(cand_ids)
    target = torch.tensor([col[p] for _, p in rows], device=dev)
    allowed = torch.zeros(B, C, dtype=torch.bool, device=dev)
    for i, (q, p) in enumerate(rows):
        if in_batch:
            allowed[i] = True
        else:
            allowed[i, list(own_negs[i])] = True
        for c in gold[q]:                                            # other relevant cases are not negatives
            if c in col and c != p:
                allowed[i, col[c]] = False
        if q in col:
            allowed[i, col[q]] = False
        allowed[i, col[p]] = True
    return allowed, target

def score_loss(S, allowed, target, loss_type, tau, margin):
    """Ranking loss on a (B, C) similarity matrix."""
    if loss_type == "infonce":
        return F.cross_entropy((S / tau).masked_fill(~allowed, -1e4), target)
    B = S.size(0)
    s_pos = S.gather(1, target[:, None])
    neg = allowed.clone(); neg[torch.arange(B), target] = False
    h = F.relu(margin - s_pos + S) * neg
    return (h.sum(1) / neg.sum(1).clamp_min(1)).mean()

def batch_loss(qE, qM, cE, cM, rows, cand_ids, own_negs, gold, sim, win_agg, loss_type, tau, margin, in_batch):
    S = similarity(qE, qM, cE, cM, sim, win_agg)                     # (B, C)
    allowed, target = batch_mask(rows, cand_ids, own_negs, gold, in_batch, S.device)
    return score_loss(S, allowed, target, loss_type, tau, margin)


def _rng_state():
    return (torch.get_rng_state(), torch.cuda.get_rng_state() if torch.cuda.is_available() else None)

def _set_rng(st):
    torch.set_rng_state(st[0])
    if st[1] is not None:
        torch.cuda.set_rng_state(st[1])

def step_backward(enc, q_cases, c_cases, loss_fn, chunk, scaler):
    """Forward + backward for one batch. chunk = cases per forward (None/0 = all at once).
    Returns the loss value. With chunk, GradCache: identical loss/gradients, much less memory."""
    cases = q_cases + c_cases
    nq = len(q_cases)
    ac = dict(device_type="cuda", dtype=torch.float16, enabled=_AUTOCAST)
    if not chunk or chunk >= len(cases):
        with torch.autocast(**ac):
            E, M = enc(cases)
        loss = loss_fn(E[:nq], M[:nq], E[nq:], M[nq:])
        scaler.scale(loss).backward()
        return loss.item()
    parts = [cases[i:i + chunk] for i in range(0, len(cases), chunk)]
    states, Es, Ms = [], [], []
    with torch.no_grad():
        for p in parts:
            states.append(_rng_state())
            with torch.autocast(**ac):
                e, m = enc(p)
            Es.append(e.float()); Ms.append(m)
    W = max(e.size(1) for e in Es)
    E = torch.cat([F.pad(e, (0, 0, 0, W - e.size(1))) for e in Es]).requires_grad_(True)
    M = torch.cat([F.pad(m, (0, W - m.size(1))) for m in Ms])
    loss = loss_fn(E[:nq], M[:nq], E[nq:], M[nq:])
    loss.backward()
    G = E.grad
    after = _rng_state()
    i = 0
    for p, st in zip(parts, states):
        _set_rng(st)
        with torch.autocast(**ac):
            e, _ = enc(p)
        g = G[i:i + len(p), :e.size(1)]
        scaler.scale((e.float() * g).sum()).backward()
        i += len(p)
    _set_rng(after)
    return loss.item()


# -------------------------------------------------------------------------------------
# Training loop: AdamW on trainable params (fp16 autocast + GradScaler), best epoch by dev Micro-F1@7.
# Saves best_trainable.pt and last_checkpoint.pt (resume: trainable, best, opt, scaler, sched, epoch, log).
# -------------------------------------------------------------------------------------
def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_model(enc, pieces, lr, epochs, batch_size, weight_decay, sim, win_agg, loss_type, tau, margin,
                in_batch, hard_source, n_hard, hard_top, pos_sampling, grad_clip=None, lr_lambda=None,
                chunk=None, batch_windows=64, save_path="best_trainable.pt", last_path=None,
                resume_from=None, max_epoch_minutes=None, hooks=None):
    """pieces = {"train_q", "train_c", "dev_q", "dev_c"}: case id -> windows.
    hooks (optional, e.g. the AdaLoRA allocator): extra_loss(), after_backward() (unscaled grads),
    after_step(), postfix(), budget, rank_pattern(), state_dict() / load_state_dict()."""
    params = [p for p in enc.parameters() if p.requires_grad]
    assert params, "no trainable parameters"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=_AUTOCAST)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    best, log, start = -1.0, [], 0
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        enc.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"):
            sched.load_state_dict(ck["sched"])
        best, log, start = ck["best"], ck["log"], ck["epoch"]
        if hooks is not None and ck.get("hooks") is not None:
            hooks.load_state_dict(ck["hooks"])
        torch.save(ck["best_trainable"], save_path)
        print(f"resumed after epoch {start} (best dev Micro-F1@7 {best:.4f})")
        del ck
    gold = GOLD["train"]
    hard_rank = None
    if hard_source == "bm25" and n_hard:
        if "train" not in BM25_RANK:
            BM25_RANK["train"] = bm25_rank("train", top=max(200, hard_top))
        hard_rank = BM25_RANK["train"]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start, epochs):
        if hard_source == "model" and n_hard:
            hard_rank = rank_split(enc, "train", pieces["train_q"], pieces["train_c"], sim, win_agg,
                                   batch_windows, top=hard_top + 50, q_ids=TRAIN_QIDS)
        enc.train()
        rows = make_epoch_rows(pos_sampling, gold)
        batches = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
        t0, tot, seen = time.time(), 0.0, 0
        pbar = tqdm(batches, desc=f"epoch {ep + 1}/{epochs}")
        for batch in pbar:
            negs = [pick_negatives(q, n_hard, hard_source, hard_rank, hard_top, gold) for q, _ in batch]
            cand_ids = list(dict.fromkeys([p for _, p in batch] + [c for n in negs for c in n]))
            col = {c: j for j, c in enumerate(cand_ids)}
            own = [{col[c] for c in n} for n in negs]
            q_cases = [pieces["train_q"][q] for q, _ in batch]
            c_cases = [pieces["train_c"][c] for c in cand_ids]
            loss_fn = lambda qE, qM, cE, cM: batch_loss(qE, qM, cE, cM, batch, cand_ids, own, gold, sim, win_agg,
                                                        loss_type, tau, margin, in_batch)
            opt.zero_grad(set_to_none=True)
            l = step_backward(enc, q_cases, c_cases, loss_fn, chunk, scaler)
            if hooks is not None:                                   # e.g. AdaLoRA orthogonality penalty
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=_AUTOCAST):
                    xl = hooks.extra_loss()
                if torch.is_tensor(xl) and xl.requires_grad:
                    scaler.scale(xl.float()).backward()
                    l += xl.item()
            if grad_clip or hooks is not None:
                scaler.unscale_(opt)                                 # hooks see real (unscaled) gradients
            if hooks is not None:
                hooks.after_backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched:
                sched.step()
            if hooks is not None:
                hooks.after_step()
            tot += l; seen += 1
            pf = dict(loss=f"{tot / seen:.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")
            if hooks is not None:
                pf.update(hooks.postfix())
            pbar.set_postfix(**pf)
            if max_epoch_minutes and time.time() - t0 > 60 * max_epoch_minutes:
                print(f"  time cap: stopped epoch after {seen}/{len(batches)} batches")
                break
        dev_rank = rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], sim, win_agg, batch_windows)
        r = evaluate_ranking(dev_rank, GOLD["dev"], show=False)
        row = {"epoch": ep + 1, "train_loss": tot / max(1, seen), "batches": seen,
               "minutes": (time.time() - t0) / 60, **{k: r[k] for k in ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")}}
        if hooks is not None:
            row.update({"rank_budget": hooks.budget, "ranks_kept": sum(hooks.rank_pattern())})
        log.append(row)
        improved = r["micro_f1"] > best
        if improved:
            best = r["micro_f1"]
            torch.save(trainable_state(enc), save_path)
        print(f"epoch {ep + 1}: train loss {row['train_loss']:.4f} | dev Micro-F1@7 {r['micro_f1']:.4f} "
              f"(P {r['micro_p']:.4f} R {r['micro_r']:.4f}) | R@100 {r['recall@100']:.4f} | MRR {r['mrr']:.4f}"
              f"{' *best*' if improved else ''} | {row['minutes']:.1f} min"
              + (f" | rank budget {hooks.budget}, kept {row['ranks_kept']}" if hooks is not None else ""))
        if last_path:
            torch.save({"trainable": trainable_state(enc), "best_trainable": torch.load(save_path),
                        "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": ep + 1, "best": best,
                        "log": log, "hooks": hooks.state_dict() if hooks is not None else None}, last_path)
    return log, best


# -------------------------------------------------------------------------------------
# Error breakdown helpers (numbers for the report; the analysis itself is yours)
# -------------------------------------------------------------------------------------
def breakdown(res, split="dev"):
    """Micro-F1@7 by gold-set size and by query length (c-6 tokens)."""
    pq = res["per_query"]
    def show(title, key, edges):
        print(title)
        for lo, hi in zip(edges[:-1], edges[1:]):
            qs = [q for q in pq if lo <= key(q) < hi]
            if qs:
                tp = sum(pq[q]["hits"] for q in qs); ng = sum(pq[q]["gold"] for q in qs)
                P = tp / (TOPK * len(qs)); R = tp / max(1, ng)
                print(f"  [{lo}, {hi}): {len(qs):4d} queries | Micro-F1@7 {2*P*R/(P+R) if P+R else 0:.4f}")
    show("by gold-set size:", lambda q: pq[q]["gold"], [0, 1, 3, 6, 8, 12, 10**9])
    L = {q: len(queries[split][q]["ids"]) for q in pq}
    e = [0] + [int(np.percentile(list(L.values()), p)) for p in (25, 50, 75)] + [10**9]
    show(f"by query length (quartiles, c-1 tokens):", lambda q: L[q], sorted(set(e)))

def _self_checks():
    lm.eval()
    q = next(iter(queries["dev"].values()))
    ids = q["ids"][:48]
    x = torch.tensor([ids], device=device)
    with torch.no_grad():
        ref = lm(x).float()                                           # Phase 1 DecoderLM.forward
        e1 = (lm.head(hidden(lm, x)).float() - ref).abs().max().item()
        e2 = (hidden(lm, x) - features(x)).abs().max().item()         # Cell B path
        # a short window batched (right-padded) with a longer one: same pooled vectors as alone
        short = ids[:30]
        for pool in POOLS:
            _e = PCREncoder(lm, pool)
            E, M = _e([[short], [ids]])
            alone, _ = _e([[short]])
            e3 = (E[0, 0] - alone[0, 0]).abs().max().item()
            assert e3 < 1e-3, f"padded pooling ({pool}) differs from the unpadded window"
        last = PCREncoder(lm, "last")([[short]])[0][0, 0]
        e4 = (last - features(torch.tensor([short], device=device))[0, -1]).abs().max().item()
    print(f"self-checks max |diff|: fused decoder {e1:.1e} | Cell B features {e2:.1e} | padded pooling "
          f"{e3:.1e} | last-token vector {e4:.1e}  (all should be ~0)")
    assert max(e1, e2, e4) < 1e-3, "decoder path differs from the Phase 1 model"

_self_checks()
TRAIN_POOL_IDS = list(pools["train"])


self-checks max |diff|: fused decoder 6.5e-05 | Cell B features 9.5e-07 | padded pooling 7.2e-07 | last-token vector 4.8e-07  (all should be ~0)


## Cell E1 — BitFit (fill section [H])

In [10]:
# =====================================================================================
# CELL E1 — BitFit (bias-only fine-tuning, Ben Zaken et al. 2022) on the c-1 decoder
#           for Phase 7 Prior Case Retrieval (bi-encoder, top-7 ranking)
#
# BitFit: every weight matrix, embedding and LayerNorm scale of Phase 1 c-1 stays frozen;
# only (a chosen subset of) the BIAS terms train. No new layers, no head: the case vectors are
# the Phase 1 decoder's own hidden states (LM head skipped, pooled in Cell E0).
# Same method as your Phase 4 / Phase 5 c-1 BitFit cells, now on the retrieval objective.
#
# Bias terms c-1 has (from your Phase 1 code):
#   attention q/k/v/out projections: bias=False  -> NO attention biases exist
#   per block (8) : ffn.net[0] (d_ff), ffn.net[2] (1024), ln1, ln2 (1024 each)
#   final         : ln_f        (the LM head has no bias and is not used here)
# The listing printed below gives the exact counts per group and block.
#
# NOTE: BitFit is NOT on the Phase 7 technique list (full FT, partial/layer-wise FT, QLoRA,
# LORMA, prefix, prompt, IA3, KD). A run from this cell is a valid c-1 submission and a
# comparison point, but it does not count as one of the 3 required techniques.
#
# Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# (fresh `lm`, nothing left trainable from an earlier run).
#
# >>> FILL IN: every value marked  # <<< FILL  in section [H]. Nothing else needs changing. <<<
# Sections marked [ANCILLARY] are plumbing.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — YOURS (copied from your c-3 BitFit run)
# -------------------------------------------------------------------------------------
# long case -> windows (same for queries and candidates)
WINDOW        = 512    # <<< FILL  tokens per window (<= 2048)
STRIDE        = 256    # <<< FILL  step between window starts (<= WINDOW)
DOC_REPR      = "spread" # <<< FILL  "all", "first", "last", "first_last" or "spread"
MAX_WINDOWS   = 8      # <<< FILL  windows kept per case (needed unless DOC_REPR == "all")

# representation and similarity
POOL          = "mean" # <<< FILL  token -> window vector: "mean", "max" or "last" (last real token; causal model)
SIMILARITY    = "cosine" # <<< FILL  "cosine", "dot" or "maxsim" (late interaction over windows)
WIN_AGG       = "mean" # <<< FILL  windows -> case vector for cosine/dot: "mean" or "max" (ignored for maxsim)

# ranking objective and negatives
LOSS          = "infonce" # <<< FILL  "infonce" or "margin"
TAU           = 0.05   # <<< FILL  infonce temperature (ignored for margin)
MARGIN        = 0.2    # <<< FILL  margin loss margin (ignored for infonce)
IN_BATCH_NEG  = True   # <<< FILL  True = other queries' candidates in the batch are negatives too
HARD_NEG      = "bm25" # <<< FILL  "none", "random", "bm25" or "model" (re-mined each epoch)
N_HARD        = 4      # <<< FILL  hard negatives per query per step (0 = none)
HARD_TOP      = 50     # <<< FILL  bm25/model: sample negatives from this many top-ranked non-relevant cases
POS_SAMPLING  = "one_per_query" # <<< FILL  "one_per_query" (1 random positive per query per epoch) or "all_pairs"

# BitFit  (the selection itself; group sizes are printed below)
BITFIT_GROUPS = ("ffn_in", "ffn_out", "ln1", "ln2", "ln_f") # <<< FILL  tuple, any of "ffn_in", "ffn_out", "ln1", "ln2", "ln_f" (final LayerNorm)
BITFIT_BLOCKS = tuple(range(8)) # <<< FILL  tuple of block indices whose biases train, from range(8), e.g. tuple(range(8))
                       #           (ln_f is outside every block and follows BITFIT_GROUPS only)

# training
LR            = 1e-4   # <<< FILL
EPOCHS        = 6      # <<< FILL  (total; with RESUME_FROM, the epochs already done count)
BATCH_SIZE    = 16     # <<< FILL  queries per optimizer step
WEIGHT_DECAY  = 0.0    # <<< FILL
GRAD_CLIP     = 1.0    # <<< FILL  max grad norm (0 = off)
LR_LAMBDA     = None   # optional: step -> LR multiplier (None = constant LR)
MAX_EPOCH_MINUTES = None   # optional time cap per epoch (None = no cap)
SEED          = 42     # <<< FILL

# [ANCILLARY] GPU memory / speed: same maths and same updates, only slower or faster.
GRAD_CHECKPOINT = True     # True = recompute each block in backward instead of storing it
CHUNK           = 4        # e.g. 8 = GradCache, encode 8 cases at a time with gradients (None = whole batch at once)
ENC_BATCH_WINDOWS = 64     # windows per forward when encoding pools for ranking (lower it on OOM)
# [ANCILLARY] continue an earlier run: path to its last_checkpoint.pt (attach that notebook's output as input;
# find it with  !find /kaggle/input -name last_checkpoint.pt ). None = fresh start. Keep every [H] value the same.
RESUME_FROM     = None
# [ANCILLARY] save the final model's rankings + scores (train/dev/test, top 100) for a KD notebook later
DUMP_TEACHER    = True

_required = ["WINDOW", "STRIDE", "DOC_REPR", "POOL", "SIMILARITY", "LOSS", "IN_BATCH_NEG", "HARD_NEG",
             "N_HARD", "POS_SAMPLING", "BITFIT_GROUPS", "BITFIT_BLOCKS", "LR", "EPOCHS", "BATCH_SIZE",
             "WEIGHT_DECAY", "GRAD_CLIP", "SEED"]
_required += ["TAU"] if LOSS == "infonce" else ["MARGIN"] if LOSS == "margin" else []
_required += ["WIN_AGG"] if SIMILARITY in ("cosine", "dot") else []
_required += ["HARD_TOP"] if HARD_NEG in ("bm25", "model") and N_HARD else []
_required += ["MAX_WINDOWS"] if DOC_REPR not in (None, "all") else []
_unset = [n for n in _required if globals()[n] is None]

BIAS_GROUPS = ("ffn_in", "ffn_out", "ln1", "ln2", "ln_f")


# -------------------------------------------------------------------------------------
# [ANCILLARY] list every bias of c-1 by (group, block) so you can choose BITFIT_GROUPS / BITFIT_BLOCKS
# -------------------------------------------------------------------------------------
_KIND = {".ffn.net.0.bias": "ffn_in", ".ffn.net.2.bias": "ffn_out", ".ln1.bias": "ln1", ".ln2.bias": "ln2"}

def bias_where(name):
    """c-1 bias parameter name -> (group, block index); block is None for ln_f."""
    if name == "ln_f.bias":
        return "ln_f", None
    kind = next((k for suf, k in _KIND.items() if name.endswith(suf)), None)
    assert name.startswith("blocks.") and kind, f"unexpected bias {name}"
    return kind, int(name.split(".")[1])

def model_biases(backbone):
    """{param name: (group, block, tensor)} for every bias of the c-1 decoder."""
    return {n: (*bias_where(n), p) for n, p in backbone.named_parameters() if n.endswith(".bias")}

_all_b = model_biases(lm)
_tab = defaultdict(int)
for _n, (_g, _s, _p) in _all_b.items():
    _tab[(_g, "-" if _s is None else _s)] += _p.numel()
print(f"c-1 has {len(_all_b)} bias tensors, {sum(_tab.values()):,} values "
      f"(of {sum(p.numel() for p in lm.parameters())/1e6:.2f}M parameters). By group (per block):")
for _g in BIAS_GROUPS:
    _row = {s: v for (g, s), v in _tab.items() if g == _g}
    _per = sorted(set(_row.values()))
    print(f"  {_g:8s} {sum(_row.values()):8,d}   " + (f"{_per[0]:,} per block x {len(_row)} blocks" if _g != "ln_f" else "final LayerNorm"))
assert {g for g, _, _ in _all_b.values()} <= set(BIAS_GROUPS)

assert not _unset, f"fill in section [H] first: {_unset}"
assert STRIDE <= WINDOW <= cfg.max_seq_len and DOC_REPR in DOC_REPRS
assert POOL in POOLS and SIMILARITY in SIMILARITIES and LOSS in LOSSES
assert HARD_NEG in HARD_SOURCES and POS_SAMPLING in POS_SAMPLINGS
assert SIMILARITY == "maxsim" or WIN_AGG in WIN_AGGS
assert IN_BATCH_NEG or (N_HARD and HARD_NEG != "none"), "no negatives at all: turn on IN_BATCH_NEG or hard negatives"
assert isinstance(BITFIT_GROUPS, (tuple, list)) and BITFIT_GROUPS, "BITFIT_GROUPS must be a non-empty tuple"
assert all(g in BIAS_GROUPS for g in BITFIT_GROUPS), f"BITFIT_GROUPS: choose from {BIAS_GROUPS}"
assert isinstance(BITFIT_BLOCKS, (tuple, list, range)), "BITFIT_BLOCKS must be a tuple of block indices (may be empty if only ln_f)"
BITFIT_BLOCKS = tuple(BITFIT_BLOCKS)
assert all(isinstance(i, int) and 0 <= i < cfg.n_layers for i in BITFIT_BLOCKS), f"BITFIT_BLOCKS: indices in range({cfg.n_layers})"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
RANKING = dict(sim=SIMILARITY, win_agg=WIN_AGG)
OBJ = dict(loss_type=LOSS, tau=TAU, margin=MARGIN, in_batch=IN_BATCH_NEG, hard_source=HARD_NEG,
           n_hard=N_HARD, hard_top=HARD_TOP, pos_sampling=POS_SAMPLING)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c1_bitfit_pcr_{SIMILARITY}_{LOSS}_{HARD_NEG}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
LAST = f"{OUT_DIR}/last_checkpoint.pt"


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze c-1, unfreeze the chosen biases, check nothing else trains
# -------------------------------------------------------------------------------------
GRAD_CHECKPOINT = GRAD_CHECKPOINT       # read by hidden() in E0
lm.eval()
for p in lm.parameters():
    p.requires_grad = False
CHOSEN = [n for n, (g, s, _) in _all_b.items() if g in BITFIT_GROUPS and (s is None or s in BITFIT_BLOCKS)]
for n in CHOSEN:
    _all_b[n][2].requires_grad = True
assert CHOSEN, "no bias matched: block groups need at least one block in BITFIT_BLOCKS"

enc = PCREncoder(lm, POOL).to(device)
count_params(enc)
_chosen_ids = {id(_all_b[n][2]) for n in CHOSEN}
_other = [n for n, p in enc.named_parameters() if p.requires_grad and id(p) not in _chosen_ids]
assert not _other, f"only the chosen biases should train, found {_other[:3]}"
_bit_n = sum(_all_b[n][2].numel() for n in CHOSEN)
print(f"BitFit: {len(CHOSEN)} bias tensors, {_bit_n:,} trainable values ({BITFIT_GROUPS} in blocks {BITFIT_BLOCKS})")

# fingerprint of every FROZEN tensor, to prove after training that only the chosen biases moved
_frozen_fp = {n: p.detach().double().sum().item() for n, p in lm.named_parameters() if not p.requires_grad}
_bias_start = {n: _all_b[n][2].detach().clone() for n in CHOSEN}

CFG_OUT = {"run": RUN_NAME, "method": "BitFit (bias-only), c-1 decoder", "build": BUILD, "pool": POOL,
           "ranking": RANKING, "objective": OBJ, "bitfit_groups": list(BITFIT_GROUPS),
           "bitfit_blocks": list(BITFIT_BLOCKS), "trainable_values": _bit_n}


# -------------------------------------------------------------------------------------
# [ANCILLARY] windows for every query and candidate
# -------------------------------------------------------------------------------------
pieces = {"train_q": build_pieces({q: queries["train"][q] for q in TRAIN_QIDS}, **BUILD, desc="train queries"),
          "train_c": build_pieces(pools["train"], **BUILD, desc="train pool"),
          "dev_q": build_pieces(queries["dev"], **BUILD, desc="dev queries"),
          "dev_c": build_pieces(pools["dev"], **BUILD, desc="dev pool"),
          "test_q": build_pieces(queries["test"], **BUILD, desc="test queries"),
          "test_c": build_pieces(pools["test"], **BUILD, desc="test pool")}


# -------------------------------------------------------------------------------------
# [ANCILLARY] GradCache self-check: chunked gradients == one-pass gradients (eval mode, tiny batch)
# -------------------------------------------------------------------------------------
if CHUNK:
    _rows = [(q, sorted(GOLD["train"][q])[0]) for q in TRAIN_QIDS[:3]]
    _cids = list(dict.fromkeys([p for _, p in _rows] + TRAIN_POOL_IDS[:3]))
    _short = lambda ws: [w[:32] for w in ws[:2]]
    _qc = [_short(pieces["train_q"][q]) for q, _ in _rows]
    _cc = [_short(pieces["train_c"][c]) for c in _cids]
    _own = [set() for _ in _rows]
    _lf = lambda qE, qM, cE, cM: batch_loss(qE, qM, cE, cM, _rows, _cids, _own, GOLD["train"], SIMILARITY,
                                            WIN_AGG, LOSS, TAU, MARGIN, True)
    _params = [p for p in enc.parameters() if p.requires_grad]
    _zero_init = []          # adapter tensors that start at 0 (made non-zero for the check, then reset)
    with torch.no_grad():
        for p in _zero_init:
            p.normal_(0, 0.02)
    _sc = torch.cuda.amp.GradScaler(enabled=False)
    enc.eval()
    _AUTOCAST = False        # compare in fp32: without loss scaling, fp16 bias/adapter gradients underflow in
                             # this tiny check and the two paths lose different amounts (training uses GradScaler)
    grads = []
    for ch in (None, 2):
        for p in _params:
            p.grad = None
        step_backward(enc, _qc, _cc, _lf, ch, _sc)
        grads.append(torch.cat([p.grad.flatten().float() for p in _params]))
    _rel = ((grads[0] - grads[1]).norm() / grads[0].norm().clamp_min(1e-12)).item()
    _AUTOCAST = torch.cuda.is_available()
    print(f"GradCache check (fp32): relative gradient difference {_rel:.2e} (should be ~1e-5 or less)")
    assert _rel < 5e-2, "chunked gradients differ from one-pass gradients"
    with torch.no_grad():
        for p in _zero_init:
            p.zero_()
        for p in _params:
            p.grad = None
    del grads, _params, _zero_init


# -------------------------------------------------------------------------------------
# [ANCILLARY] zero-shot Phase 1 model (no bias changed yet == untouched c-1 decoder) on dev, for the report
# -------------------------------------------------------------------------------------
print("\n-- dev before training (Phase 1 c-1 decoder as-is)")
zero_shot = evaluate_ranking(rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING,
                                        batch_windows=ENC_BATCH_WINDOWS), GOLD["dev"], name="zero-shot dev")
print(f"BM25 dev Micro-F1@7 {bm25_dev['micro_f1']:.4f}")


# -------------------------------------------------------------------------------------
# [ANCILLARY] training, dev scoring, test submission
# -------------------------------------------------------------------------------------
t_train = time.time()
log, best = train_model(enc, pieces, lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
                        **RANKING, **OBJ, grad_clip=GRAD_CLIP, lr_lambda=LR_LAMBDA, chunk=CHUNK,
                        batch_windows=ENC_BATCH_WINDOWS, save_path=SAVE, last_path=LAST,
                        resume_from=RESUME_FROM, max_epoch_minutes=MAX_EPOCH_MINUTES)
train_min = (time.time() - t_train) / 60
train_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0

enc.load_state_dict(torch.load(SAVE), strict=False)
_named = dict(lm.named_parameters())
_moved = [n for n, v in _frozen_fp.items() if _named[n].detach().double().sum().item() != v]
assert not _moved, f"a frozen tensor changed during training: {_moved[:3]}"
_delta = max((_all_b[n][2].detach() - _bias_start[n]).abs().max().item() for n in CHOSEN)
print(f"frozen weights unchanged | largest bias change {_delta:.2e}" + ("  (0 = no batch trained)" if _delta == 0 else ""))
t0 = time.time()
dev_rank, dev_scores = rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING,
                                  batch_windows=ENC_BATCH_WINDOWS, top=100, return_scores=True)
dev_s = time.time() - t0
print("\n-- dev (best epoch)")
res = evaluate_ranking(dev_rank, GOLD["dev"], name="BitFit dev")
breakdown(res)
print(f"zero-shot dev Micro-F1@7 {zero_shot['micro_f1']:.4f} | BM25 {bm25_dev['micro_f1']:.4f} | "
      f"BitFit {res['micro_f1']:.4f}")
print(f"training {train_min:.1f} min | dev ranking {dev_s:.0f} s ({len(dev_rank)} queries x {len(pools['dev'])} "
      f"candidates) | peak training memory {train_mem:.2f} GB | trainable values {_bit_n:,}")
_metrics = ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")
json.dump({**CFG_OUT, "log": log, "train_minutes": train_min, "peak_mem_gb": train_mem, "dev_rank_seconds": dev_s,
           "dev": {k: res[k] for k in _metrics},
           "zero_shot_dev": {k: zero_shot[k] for k in _metrics},
           "bm25_dev": {k: bm25_dev[k] for k in _metrics},
           "dev_rank": {q: v[:20] for q, v in dev_rank.items()}}, open(f"{OUT_DIR}/val_log.json", "w"))

# Test ranking with the best weights -> /kaggle/working/<RUN_NAME>/submission.zip
test_rank, test_scores = rank_split(enc, "test", pieces["test_q"], pieces["test_c"], **RANKING,
                                    batch_windows=ENC_BATCH_WINDOWS, top=100, return_scores=True)
write_prediction(test_rank, out_dir=OUT_DIR)

# [ANCILLARY] everything a KD notebook needs: rankings + similarity scores, and how to rebuild this model
if DUMP_TEACHER:
    tr_rank, tr_scores = rank_split(enc, "train", pieces["train_q"], pieces["train_c"], **RANKING,
                                    batch_windows=ENC_BATCH_WINDOWS, top=100, q_ids=TRAIN_QIDS, return_scores=True)
    json.dump({s: {q: {"ids": r[q], "scores": [round(x, 5) for x in sc[q]]} for q in r}
               for s, r, sc in (("train", tr_rank, tr_scores), ("dev", dev_rank, dev_scores),
                                ("test", test_rank, test_scores))},
              open(f"{OUT_DIR}/teacher_rankings.json", "w"))
json.dump({**CFG_OUT, "weights": "best_trainable.pt"}, open(f"{OUT_DIR}/teacher_config.json", "w"), indent=1)
print("files for later runs / KD:", sorted(os.listdir(OUT_DIR)))


c-1 has 33 bias tensors, 58,368 values (of 110.98M parameters). By group (per block):
  ffn_in     32,768   4,096 per block x 8 blocks
  ffn_out     8,192   1,024 per block x 8 blocks
  ln1         8,192   1,024 per block x 8 blocks
  ln2         8,192   1,024 per block x 8 blocks
  ln_f        1,024   final LayerNorm
trainable 0.058M / total 110.98M (0.05%)
BitFit: 33 bias tensors, 58,368 trainable values (('ffn_in', 'ffn_out', 'ln1', 'ln2', 'ln_f') in blocks (0, 1, 2, 3, 4, 5, 6, 7))


train queries:   0%|          | 0/817 [00:00<?, ?it/s]

train queries: 817 cases | windows per case mean 7.69 max 8 | tokens seen 42.4%


train pool:   0%|          | 0/4320 [00:00<?, ?it/s]

train pool: 4320 cases | windows per case mean 7.28 max 8 | tokens seen 43.7%


dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

... (54 outputs left out)

encode train pool:   0%|          | 0/4320 [00:00<?, ?it/s]

encode train queries:   0%|          | 0/817 [00:00<?, ?it/s]

rank train:   0%|          | 0/4 [00:00<?, ?it/s]

files for later runs / KD: ['best_trainable.pt', 'last_checkpoint.pt', 'prediction.json', 'submission.zip', 'teacher_config.json', 'teacher_rankings.json', 'val_log.json']
